<a href="https://colab.research.google.com/github/sametz/BCCE2026/blob/main/eq_ax_cyMe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install rdkit py3Dmol

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 36.7 MB/s eta 0:00:00


In [13]:
import rdkit.Chem as Chem
import rdkit.Chem.AllChem as AllChem
import py3Dmol
import numpy as np

# Define the SMILES string for methylcyclohexane
smiles = 'CC1CCCCC1'
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol) # Add hydrogens to get proper 3D geometry

# Generate multiple conformers
num_confs = 500 # Increased number of conformers to improve chances of finding good ones
# Use a consistent random seed for reproducibility of the initial conformer set
AllChem.EmbedMultipleConfs(mol, numConfs=num_confs, randomSeed=42, numThreads=0)
# Optimize all generated conformers using UFF
AllChem.UFFOptimizeMoleculeConfs(mol)

# Identify the methyl carbon (atom index 0) and the ring carbon it's attached to (atom index 1)
methyl_carbon_idx = 0
ring_attachment_idx = 1

# Helper function to classify the methyl group as axial or equatorial
def get_methyl_orientation_dot_product(mol_obj, conformer):
    # Get positions of methyl carbon and its attachment point
    methyl_pos = conformer.GetAtomPosition(methyl_carbon_idx)
    attachment_pos = conformer.GetAtomPosition(ring_attachment_idx)

    # Get the neighbors of the attachment point (excluding the methyl carbon)
    attachment_atom = mol_obj.GetAtomWithIdx(ring_attachment_idx)
    ring_neighbors_of_attachment = [
        a.GetIdx() for a in attachment_atom.GetNeighbors() if a.GetIdx() != methyl_carbon_idx
    ]

    if len(ring_neighbors_of_attachment) < 2:
        # Should have at least two ring neighbors for a cyclohexane
        return None # Indicate inability to classify

    # Get positions of two ring neighbors of the attachment carbon
    neighbor_pos1 = conformer.GetAtomPosition(ring_neighbors_of_attachment[0])
    neighbor_pos2 = conformer.GetAtomPosition(ring_neighbors_of_attachment[1])

    # Calculate vectors for the C-C bonds of the attachment carbon within the ring
    vec1 = np.array([neighbor_pos1.x - attachment_pos.x, neighbor_pos1.y - attachment_pos.y, neighbor_pos1.z - attachment_pos.z])
    vec2 = np.array([neighbor_pos2.x - attachment_pos.x, neighbor_pos2.y - attachment_pos.y, neighbor_pos2.z - attachment_pos.z])

    # Calculate the cross product to get a vector approximately normal to the local ring plane at the attachment point.
    # This vector approximates the axial direction of the C-H bonds at this carbon.
    local_normal_vector = np.cross(vec1, vec2)
    # Handle cases where cross product might be zero (e.g., linear geometry, highly distorted ring)
    if np.linalg.norm(local_normal_vector) == 0:
        return None # Cannot determine normal vector
    local_normal_vector = local_normal_vector / np.linalg.norm(local_normal_vector) # Normalize the normal vector

    # Vector from attachment point to methyl carbon
    vec_methyl = np.array([methyl_pos.x - attachment_pos.x, methyl_pos.y - attachment_pos.y, methyl_pos.z - attachment_pos.z])
    if np.linalg.norm(vec_methyl) == 0:
        return None # Cannot determine methyl vector
    vec_methyl = vec_methyl / np.linalg.norm(vec_methyl) # Normalize the methyl vector

    # Calculate the dot product to see how parallel vec_methyl is to the local_normal_vector
    # A dot product close to 1 or -1 means parallel (axial).
    # A dot product close to 0 means perpendicular (equatorial).
    dot_product = np.dot(vec_methyl, local_normal_vector)
    return dot_product

# Collect all conformers and their calculated dot products
# We only care about conformers where a dot product could be calculated
all_valid_conformer_data = []
for i in range(mol.GetNumConformers()):
    conf = mol.GetConformer(i)
    dp_value = get_methyl_orientation_dot_product(mol, conf)

    if dp_value is not None:
        all_valid_conformer_data.append((conf, dp_value))

axial_conf = None
equatorial_conf = None

if not all_valid_conformer_data:
    print("Error: No valid conformers found for analysis. Displaying the first conformer for both.")
    # Fallback to default if no valid data
    equatorial_conf = mol.GetConformer(0)
    axial_conf = mol.GetConformer(0)
elif len(all_valid_conformer_data) < 2: # Need at least two distinct conformers
    print("Warning: Only one or zero distinct valid conformers found. Using it/first for both displays.")
    axial_conf = all_valid_conformer_data[0][0] if all_valid_conformer_data else mol.GetConformer(0)
    equatorial_conf = axial_conf # Use the same if only one distinct conformer
else:
    # Sort all valid conformers by absolute dot product in descending order (most axial first)
    sorted_by_axial_character = sorted(all_valid_conformer_data, key=lambda x: abs(x[1]), reverse=True)

    # The first one is the 'most axial' conformer
    axial_conf_tuple = sorted_by_axial_character[0]
    axial_conf = axial_conf_tuple[0]
    print(f"Selected axial conformer (Conf ID: {axial_conf.GetId()}, Dot Product: {axial_conf_tuple[1]:.2f})")

    # Filter out the selected axial conformer to find a distinct equatorial one
    remaining_for_equatorial = [item for item in sorted_by_axial_character if item[0].GetId() != axial_conf.GetId()]

    if not remaining_for_equatorial:
        print("Warning: No distinct conformer found for equatorial after selecting axial. Using the axial conformer for equatorial display as well.")
        equatorial_conf = axial_conf # Fallback if only one truly distinct conformer
    else:
        # Sort the remaining by absolute dot product in ascending order (most equatorial first)
        sorted_by_equatorial_character = sorted(remaining_for_equatorial, key=lambda x: abs(x[1]))
        equatorial_conf_tuple = sorted_by_equatorial_character[0]
        equatorial_conf = equatorial_conf_tuple[0]
        print(f"Selected equatorial conformer (Conf ID: {equatorial_conf.GetId()}, Dot Product: {equatorial_conf_tuple[1]:.2f})")

# Create new RDKit molecule objects for display, each with only its selected conformer
mol_equatorial_display = Chem.Mol(mol)
mol_equatorial_display.RemoveAllConformers()
mol_equatorial_display.AddConformer(equatorial_conf, assignId=True)

mol_axial_display = Chem.Mol(mol)
mol_axial_display.RemoveAllConformers()
mol_axial_display.AddConformer(axial_conf, assignId=True)

# Create a 3Dmol viewer with a 1x2 grid for side-by-side display
view = py3Dmol.view(query='_ALL_', linked=False, viewergrid=(1, 2))

# Add the equatorial molecule to the first panel
# Convert RDKit molecule to MolBlock string for py3Dmol
view.addModel(Chem.MolToMolBlock(mol_equatorial_display), 'mol', viewer=(0,0))
view.setStyle({'stick':{}}, viewer=(0,0))
view.zoomTo(viewer=(0,0))
view.addLabel('Equatorial Conformation', {'position': {'x': 0, 'y': 0, 'z': 0}, 'fontColor': 'black', 'fontSize': 14, 'backgroundColor': 'rgba(255, 255, 255, 0.7)'}, {'viewer': [0, 0]})

# Add the axial molecule to the second panel
view.addModel(Chem.MolToMolBlock(mol_axial_display), 'mol', viewer=(0,1))
view.setStyle({'stick':{}}, viewer=(0,1))
view.zoomTo(viewer=(0,1))
view.addLabel('Axial Conformation', {'position': {'x': 0, 'y': 0, 'z': 0}, 'fontColor': 'black', 'fontSize': 14, 'backgroundColor': 'rgba(255, 255, 255, 0.7)'}, {'viewer': [0, 1]})

# Render the viewer
view.render()

Selected axial conformer (Conf ID: 223, Dot Product: -0.79)
Selected equatorial conformer (Conf ID: 428, Dot Product: -0.75)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
import rdkit.Chem as Chem
import rdkit.Chem.AllChem as AllChem
import py3Dmol
import numpy as np

# Define the SMILES string for chlorocyclohexane
smiles_chloro = 'ClC1CCCCC1'
mol_chloro = Chem.MolFromSmiles(smiles_chloro)
mol_chloro = Chem.AddHs(mol_chloro) # Add hydrogens to get proper 3D geometry

# Generate multiple conformers
num_confs = 100 # Generate more conformers to increase chances of finding axial
AllChem.EmbedMultipleConfs(mol_chloro, numConfs=num_confs, randomSeed=42, numThreads=0)
AllChem.UFFOptimizeMoleculeConfs(mol_chloro)

axial_chloro_conf = None

# Identify the chlorine atom index (atom with atomic number 17) and its attachment point
chloro_atom_idx = -1
ring_attachment_idx_chloro = -1
for atom in mol_chloro.GetAtoms():
    if atom.GetAtomicNum() == 17: # Chlorine atomic number
        chloro_atom_idx = atom.GetIdx()
        # Find the carbon it's attached to (should be a ring carbon)
        for neighbor in atom.GetNeighbors():
            if neighbor.GetAtomicNum() == 6: # Carbon
                ring_attachment_idx_chloro = neighbor.GetIdx()
                break
        break

if chloro_atom_idx == -1 or ring_attachment_idx_chloro == -1:
    raise ValueError("Could not identify chlorine atom or its attachment point.")

# Helper function to classify the chlorine group as axial or equatorial
def is_axial_chlorine(mol_obj, conformer):
    chloro_pos = conformer.GetAtomPosition(chloro_atom_idx)
    attachment_pos = conformer.GetAtomPosition(ring_attachment_idx_chloro)

    attachment_atom = mol_obj.GetAtomWithIdx(ring_attachment_idx_chloro)
    ring_neighbors_of_attachment = [
        a.GetIdx() for a in attachment_atom.GetNeighbors() if a.GetAtomicNum() == 6 and a.GetIdx() != chloro_atom_idx
    ]

    # Ensure we have two ring neighbors
    if len(ring_neighbors_of_attachment) < 2:
        return False, None

    # Get positions of two ring neighbors of the attachment carbon
    neighbor_pos1 = conformer.GetAtomPosition(ring_neighbors_of_attachment[0])
    neighbor_pos2 = conformer.GetAtomPosition(ring_neighbors_of_attachment[1])

    vec1 = np.array([neighbor_pos1.x - attachment_pos.x, neighbor_pos1.y - attachment_pos.y, neighbor_pos1.z - attachment_pos.z])
    vec2 = np.array([neighbor_pos2.x - attachment_pos.x, neighbor_pos2.y - attachment_pos.y, neighbor_pos2.z - attachment_pos.z])

    local_normal_vector = np.cross(vec1, vec2)
    local_normal_vector = local_normal_vector / np.linalg.norm(local_normal_vector) # Normalize

    vec_chloro = np.array([chloro_pos.x - attachment_pos.x, chloro_pos.y - attachment_pos.y, chloro_pos.z - attachment_pos.z])
    vec_chloro = vec_chloro / np.linalg.norm(vec_chloro) # Normalize

    dot_product = np.dot(vec_chloro, local_normal_vector)

    axial_threshold = 0.7 # A relatively high threshold for clear axiality

    if abs(dot_product) > axial_threshold:
        return True, dot_product # It's axial
    else:
        return False, dot_product # Not clearly axial

# Iterate through conformers to find an axial chlorine conformer
for i in range(mol_chloro.GetNumConformers()):
    conf = mol_chloro.GetConformer(i)
    is_axial, dp = is_axial_chlorine(mol_chloro, conf)

    if is_axial is True:
        axial_chloro_conf = conf
        print(f"Found axial conformer (Conf ID: {conf.GetId()}, Dot Product: {dp:.2f})")
        break # Found one, exit loop

# Fallback if no clear axial conformer was found by the heuristic
if axial_chloro_conf is None:
    print("Warning: Could not find a clear axial conformer for chlorocyclohexane. Displaying the first conformer.")
    axial_chloro_conf = mol_chloro.GetConformer(0)

# Create a new RDKit molecule object for display with only the selected conformer
mol_chloro_display = Chem.Mol(mol_chloro)
mol_chloro_display.RemoveAllConformers()
mol_chloro_display.AddConformer(axial_chloro_conf, assignId=True)

# Create a 3Dmol viewer for a single molecule
view_chloro = py3Dmol.view(width=400, height=400)

# Add the axial chlorocyclohexane molecule
view_chloro.addModel(Chem.MolToMolBlock(mol_chloro_display), 'mol')
view_chloro.setStyle({'stick':{}})
view_chloro.zoomTo()
view_chloro.addLabel('Axial Chlorocyclohexane', {'position': {'x': 0, 'y': 0, 'z': 0}, 'fontColor': 'black', 'fontSize': 14, 'backgroundColor': 'rgba(255, 255, 255, 0.7)'})

# Render the viewer
view_chloro.render()

Found axial conformer (Conf ID: 0, Dot Product: 0.77)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
import rdkit.Chem as Chem
import rdkit.Chem.AllChem as AllChem
import py3Dmol
import numpy as np

# Define the SMILES string for methylcyclohexane
smiles_methyl = 'CC1CCCCC1'
mol_methyl = Chem.MolFromSmiles(smiles_methyl)
mol_methyl = Chem.AddHs(mol_methyl) # Add hydrogens to get proper 3D geometry

# Generate multiple conformers
num_confs = 100 # Generate more conformers to increase chances of finding axial
AllChem.EmbedMultipleConfs(mol_methyl, numConfs=num_confs, randomSeed=42, numThreads=0)
AllChem.UFFOptimizeMoleculeConfs(mol_methyl)

axial_methyl_conf = None

# Identify the methyl carbon (atom index 0) and the ring carbon it's attached to (atom index 1)
methyl_carbon_idx = 0
ring_attachment_idx = 1

# Helper function to classify the methyl group as axial or equatorial
def is_axial_methyl_single_mol(mol_obj, conformer):
    # Get positions of methyl carbon and its attachment point
    methyl_pos = conformer.GetAtomPosition(methyl_carbon_idx)
    attachment_pos = conformer.GetAtomPosition(ring_attachment_idx)

    # Get the neighbors of the attachment point (excluding the methyl carbon)
    attachment_atom = mol_obj.GetAtomWithIdx(ring_attachment_idx)
    ring_neighbors_of_attachment = [
        a.GetIdx() for a in attachment_atom.GetNeighbors() if a.GetIdx() != methyl_carbon_idx
    ]

    if len(ring_neighbors_of_attachment) < 2:
        # Should have at least two ring neighbors for a cyclohexane
        return False, None

    # Get positions of two ring neighbors of the attachment carbon
    neighbor_pos1 = conformer.GetAtomPosition(ring_neighbors_of_attachment[0])
    neighbor_pos2 = conformer.GetAtomPosition(ring_neighbors_of_attachment[1])

    # Calculate vectors for the C-C bonds of the attachment carbon within the ring
    vec1 = np.array([neighbor_pos1.x - attachment_pos.x, neighbor_pos1.y - attachment_pos.y, neighbor_pos1.z - attachment_pos.z])
    vec2 = np.array([neighbor_pos2.x - attachment_pos.x, neighbor_pos2.y - attachment_pos.y, neighbor_pos2.z - attachment_pos.z])

    # Calculate the cross product to get a vector approximately normal to the local ring plane at the attachment point.
    # This vector approximates the axial direction of the C-H bonds at this carbon.
    local_normal_vector = np.cross(vec1, vec2)
    local_normal_vector = local_normal_vector / np.linalg.norm(local_normal_vector) # Normalize the normal vector

    # Vector from attachment point to methyl carbon
    vec_methyl = np.array([methyl_pos.x - attachment_pos.x, methyl_pos.y - attachment_pos.y, methyl_pos.z - attachment_pos.z])
    vec_methyl = vec_methyl / np.linalg.norm(vec_methyl) # Normalize the methyl vector

    # Calculate the dot product to see how parallel vec_methyl is to the local_normal_vector
    # A dot product close to 1 or -1 means parallel (axial).
    # A dot product close to 0 means perpendicular (equatorial).
    dot_product = np.dot(vec_methyl, local_normal_vector)

    # Heuristic thresholds for classification
    axial_threshold = 0.7       # If |dot_product| > 0.7, consider axial

    if abs(dot_product) > axial_threshold:
        return True, dot_product # It's axial
    else:
        return False, dot_product # Not clearly axial

# Iterate through conformers to find an axial methyl conformer
for i in range(mol_methyl.GetNumConformers()):
    conf = mol_methyl.GetConformer(i)
    is_axial, dp = is_axial_methyl_single_mol(mol_methyl, conf)

    if is_axial is True:
        axial_methyl_conf = conf
        print(f"Found axial conformer (Conf ID: {conf.GetId()}, Dot Product: {dp:.2f})")
        break # Found one, exit loop

# Fallback if no clear axial conformer was found by the heuristic
if axial_methyl_conf is None:
    print("Warning: Could not find a clear axial conformer for methylcyclohexane. Displaying the first conformer.")
    axial_methyl_conf = mol_methyl.GetConformer(0)

# Create a new RDKit molecule object for display with only the selected conformer
mol_methyl_display = Chem.Mol(mol_methyl)
mol_methyl_display.RemoveAllConformers()
mol_methyl_display.AddConformer(axial_methyl_conf, assignId=True)

# Create a 3Dmol viewer for a single molecule
view_methyl = py3Dmol.view(width=400, height=400)

# Add the axial methylcyclohexane molecule
view_methyl.addModel(Chem.MolToMolBlock(mol_methyl_display), 'mol')
view_methyl.setStyle({'stick':{}})
view_methyl.zoomTo()
view_methyl.addLabel('Axial Methylcyclohexane', {'position': {'x': 0, 'y': 0, 'z': 0}, 'fontColor': 'black', 'fontSize': 14, 'backgroundColor': 'rgba(255, 255, 255, 0.7)'})

# Render the viewer
view_methyl.render()

Found axial conformer (Conf ID: 0, Dot Product: -0.79)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.